# Neutrino oscillations

*nuclear · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/neutrino_oscillations)

Neutrinos are made in a definite flavour ($\nu_e, \nu_\mu, \nu_\tau$) but travel as three mass states with slightly different phases. The PMNS mixing matrix and one complex einsum over the mass index give the probability of every flavour change for a whole grid of baselines, showing a $\nu_\mu$ beam turning into $\nu_\tau$ and $\nu_e$ as it flies.

**Vector form**

$$P(\nu_\alpha\to\nu_\beta) = \Big|\sum_i U_{\beta i}\,U^*_{\alpha i}\,e^{-i m_i^2 L/2E}\Big|^2,\qquad U = R_{23}\,U_{13}(\delta)\,R_{12}$$

**Index (tensor) form**

$$A_{k\beta\alpha} = U_{\beta i}\,U^*_{\alpha i}\,e^{-i\,m_i^2 L_k/2E},\qquad P_{k\beta\alpha} = |A_{k\beta\alpha}|^2,\qquad U_{\alpha i} = [R_{23}]_{\alpha\beta}[U_{13}]_{\beta\gamma}[R_{12}]_{\gamma i}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $U_{\alpha i}$ | `pmns(theta12, theta13, theta23, delta)` | (3, 3) complex | flavour $\alpha$ × mass state $i$; product of three rotations via einsum('ab,bc,ci->ai') |
| $m_i^2$ | `squared_masses(dm21, dm31)` | (3,) | only the differences $\Delta m^2_{21}, \Delta m^2_{31}$ matter; $m_1^2 = 0$ |
| $L_k$ | `baselines(10, l_max, K)` | (K,) | log-spaced distances in km |
| $e^{-i m_i^2 L_k/2E}$ | `phases(m2, dist / energy)` | (K, 3) complex | phase of each mass state at each baseline ($2.534\,m^2 L/E$ in eV², km, GeV) |
| $A_{k\beta\alpha}$ | `einsum('bi,ai,ki->kba', u, u.conj(), ph)` | (K, 3, 3) complex | repeated $i$ ⇒ summed: the three mass states interfere |
| $P_{k\beta\alpha}$ | `amplitude(...).abs() ** 2` | (K, 3, 3) | all nine transition probabilities at every baseline |

Because $U$ is unitary, $\sum_\beta P(\alpha\to\beta) = 1$ at every distance: neutrinos change flavour but none are lost. With only two flavours the formula collapses to $P = \sin^2 2\theta\,\sin^2(1.267\,\Delta m^2 L/E)$. Here the large $\Delta m^2_{31}$ gives the fast "atmospheric" oscillation (first $\nu_\mu$ dip near $L/E \approx 500$ km/GeV, mostly into $\nu_\tau$ because $\theta_{23} \approx 45°$) and the small $\Delta m^2_{21}$ the slow "solar" one (near $15\,000$ km/GeV). The dots below are sample neutrinos: each shows the flavour a detector at its distance would record, drawn with the computed probabilities (fixed seed). Simplifications: vacuum oscillations only — the MSW matter effect inside the Earth or Sun is left out — and normal mass ordering. Default parameters are close to current global fits. Try $\delta_{CP}$ = 0 vs 270° for a $\nu_\mu \to \nu_e$ beam.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/neutrino_oscillations/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import numpy as np
import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import pmns


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    angles = [math.radians(params[k]) for k in ("theta12", "theta13", "theta23", "delta_cp")]
    u = pmns(*angles)  # (3, 3) complex
    m2 = tensors.squared_masses(params["dm21"] * 1e-5, params["dm31"] * 1e-3, dev)  # (3,) eV²
    dist = tensors.baselines(10.0, params["l_max"], 2500, dev)  # (K,) km
    alpha = tensors.FLAVOURS.index(params["flavour"])

    p = solve.oscillate(u, m2, dist, params["energy"], alpha)  # (K, 3)

    beam_l, beam_y, beam_u = tensors.beam(400, 10.0, params["l_max"])
    index = torch.from_numpy(np.searchsorted(dist.cpu().numpy(), beam_l.numpy()).clip(max=len(dist) - 1))
    flavour = solve.detected_flavour(p.cpu()[index], beam_u)  # (n,)

    print("PMNS |U|:")
    for row, name in zip(u.abs().tolist(), ["e  ", "mu ", "tau"]):
        print(f"  {name} " + "  ".join(f"{x:.3f}" for x in row))
    print(f"unitarity: max |U U† − 1| = {(u @ u.conj().T - torch.eye(3)).abs().max().item():.1e}")
    print(f"max |Σ_β P − 1| over all baselines = {(p.sum(1) - 1).abs().max().item():.1e}")
    for target in (300.0, 1300.0, 12700.0):
        k = int(torch.argmin((dist - target).abs()))
        probs = "  ".join(f"P({params['flavour']}→{f}) = {p[k, b].item():.3f}" for b, f in enumerate(tensors.FLAVOURS))
        print(f"L = {dist[k].item():7.0f} km   {probs}")
    return plot.animate(
        dist, p, beam_l, beam_y, flavour, alpha, params["energy"], out_dir / "neutrino_oscillations.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/neutrino_oscillations/equation.py
r"""Physics → tensor translation.

A neutrino born with flavour α (e, μ, τ) is a superposition of three mass states i = 1, 2, 3 with amplitudes
U^*_{αi}, the elements of the PMNS mixing matrix. In vacuum each mass state picks up the phase
φ_i = m_i^2 L / 2E (natural units), so the amplitude to be detected as flavour β after a distance L is

    A_{βα}(L) = U_{βi} \, U^*_{αi} \, e^{-i m_i^2 L / 2E}        (i summed),
    P(α → β) = |A_{βα}|^2.

With L in km, E in GeV and m² in eV²:  m_i^2 L / 2E = 2.534 · m_i^2 L / E.

The PMNS matrix is a product of three rotations, one of them carrying the CP phase δ:

    U = R_{23}(θ_{23}) \, U_{13}(θ_{13}, δ) \, R_{12}(θ_{12}).
"""

import torch

PHASE = 2.534  # (1/2E) m² L in units eV², km, GeV  (= 2 × 1.267)


def pmns(theta12: float, theta13: float, theta23: float, delta: float) -> torch.Tensor:
    r"""U_{αi} = [R_{23}]_{αβ} [U_{13}]_{βγ} [R_{12}]_{γi}   (β, γ summed).   angles in radians → (3, 3) complex"""
    c12, s12 = _cos_sin(theta12)
    c13, s13 = _cos_sin(theta13)
    c23, s23 = _cos_sin(theta23)
    e = torch.exp(torch.tensor(1j * delta, dtype=torch.complex128))
    zero, one = torch.zeros((), dtype=torch.complex128), torch.ones((), dtype=torch.complex128)
    r23 = torch.stack([one, zero, zero, zero, c23, s23, zero, -s23, c23]).reshape(3, 3)
    u13 = torch.stack([c13, zero, s13 / e, zero, one, zero, -s13 * e, zero, c13]).reshape(3, 3)
    r12 = torch.stack([c12, s12, zero, -s12, c12, zero, zero, zero, one]).reshape(3, 3)
    return torch.einsum("ab,bc,ci->ai", r23, u13, r12)


def _cos_sin(theta: float) -> tuple[torch.Tensor, torch.Tensor]:
    t = torch.tensor(theta, dtype=torch.float64)
    return torch.cos(t).to(torch.complex128), torch.sin(t).to(torch.complex128)


def phases(m2: torch.Tensor, l_over_e: torch.Tensor) -> torch.Tensor:
    r"""e^{-i m_i^2 L/2E} for every baseline.   m2: (3,) eV², l_over_e: (K,) km/GeV → (K, 3) complex"""
    phi = PHASE * torch.einsum("k,i->ki", l_over_e, m2)
    return torch.exp(-1j * phi)


def amplitude(u: torch.Tensor, ph: torch.Tensor) -> torch.Tensor:
    r"""A_{kβα} = U_{βi} U^*_{αi} e^{-iφ_{ki}}   (i summed).   u: (3, 3), ph: (K, 3) → (K, 3, 3)"""
    return torch.einsum("bi,ai,ki->kba", u, u.conj(), ph)


def probability(u: torch.Tensor, ph: torch.Tensor) -> torch.Tensor:
    r"""P_{kβα} = |A_{kβα}|^2: probability α → β at baseline k.   → (K, 3, 3) real"""
    return amplitude(u, ph).abs() ** 2


def two_flavour(theta: float, dm2: float, l_over_e: torch.Tensor) -> torch.Tensor:
    r"""P = \sin^2 2θ \, \sin^2(1.267 Δm^2 L/E): the textbook two-flavour formula."""
    return torch.sin(torch.tensor(2 * theta)) ** 2 * torch.sin(PHASE / 2 * dm2 * l_over_e) ** 2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/neutrino_oscillations/tensors.py
"""Build the concrete tensors: squared masses, the baseline grid and a beam of sample neutrinos."""

import math

import torch

FLAVOURS = ["e", "mu", "tau"]


def squared_masses(dm21: float, dm31: float, device: torch.device) -> torch.Tensor:
    """m_i² in eV², normal ordering with the lightest set to 0 (only differences matter): shape (3,)."""
    return torch.tensor([0.0, dm21, dm31], dtype=torch.float64, device=device)


def baselines(l_min: float, l_max: float, n: int, device: torch.device) -> torch.Tensor:
    """L_k in km, log-spaced so short and long oscillations both show up: shape (K,)."""
    return torch.logspace(math.log10(l_min), math.log10(l_max), n, dtype=torch.float64, device=device)


def beam(n: int, l_min: float, l_max: float, seed: int = 0) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """n sample neutrinos for the picture: log-uniform positions (km), a lateral offset and a uniform number
    u_n that decides which flavour a detector at that position would see: three tensors of shape (n,)."""
    gen = torch.Generator().manual_seed(seed)
    log_l = math.log10(l_min) + (math.log10(l_max) - math.log10(l_min)) * torch.rand(n, generator=gen)
    return 10**log_l, torch.rand(n, generator=gen), torch.rand(n, generator=gen)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/neutrino_oscillations/solve.py
"""Evaluate the oscillation probabilities on the whole baseline grid at once."""

import torch

from .equation import phases, probability


def oscillate(u: torch.Tensor, m2: torch.Tensor, dist: torch.Tensor, energy: float, alpha: int) -> torch.Tensor:
    """P(α → β) for a beam born as flavour α: shape (K, 3) with columns β = e, μ, τ."""
    p = probability(u.to(dist.device), phases(m2, dist / energy))  # (K, β, α)
    return p[:, :, alpha]


def detected_flavour(p_at: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    """Sample a flavour per neutrino: the first β whose cumulative probability exceeds u_n.

    p_at: (n, 3) probabilities at each neutrino's position, u: (n,) → (n,) integer flavour index.
    """
    cumulative = torch.cumsum(p_at, dim=1)  # (n, 3)
    return (cumulative < u[:, None]).sum(dim=1).clamp(max=2)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/neutrino_oscillations/plot.py
"""Top: flavour probabilities along the baseline (stacked). Bottom: a beam of neutrinos coloured by flavour."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

COLOURS = ["#2563eb", "#dc2626", "#16a34a"]  # e, μ, τ
NAMES = [r"$\nu_e$", r"$\nu_\mu$", r"$\nu_\tau$"]


def animate(
    dist: torch.Tensor,
    p: torch.Tensor,
    beam_l: torch.Tensor,
    beam_y: torch.Tensor,
    beam_flavour: torch.Tensor,
    born: int,
    energy: float,
    path: Path,
) -> Path:
    dist, p = dist.cpu().numpy(), p.cpu().numpy()  # (K,), (K, 3)
    bl, by, bf = beam_l.cpu().numpy(), beam_y.cpu().numpy(), beam_flavour.cpu().numpy()
    stacked = np.concatenate([np.zeros((len(dist), 1)), np.cumsum(p, axis=1)], axis=1)  # (K, 4) band edges

    fig, (ax_p, ax_b) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    for beta in range(3):
        ax_p.fill_between(dist, stacked[:, beta], stacked[:, beta + 1], color=COLOURS[beta], alpha=0.12, lw=0)
    bands = []
    ax_p.set_xscale("log")
    ax_p.set_xlim(dist[0], dist[-1])
    ax_p.set_ylim(0, 1)
    ax_p.set_ylabel("flavour probability")
    ax_p.set_title(f"beam born as {NAMES[born]},  E = {energy:g} GeV", fontsize=10)
    front_p = ax_p.axvline(dist[0], color="#111827", lw=1)
    readout = ax_p.text(1.01, 0.5, "", transform=ax_p.transAxes, va="center", fontsize=9)
    for beta in range(3):
        ax_p.plot([], [], color=COLOURS[beta], lw=6, label=NAMES[beta])
    ax_p.legend(loc="lower left", fontsize=8, ncol=3, framealpha=0.9)

    ax_b.set_ylim(-0.1, 1.1)
    ax_b.set_yticks([])
    ax_b.set_xlabel("distance travelled L (km)")
    ax_b.grid(False)
    scatter = ax_b.scatter([], [], s=12)
    front_b = ax_b.axvline(dist[0], color="#111827", lw=1)

    def update(step: int):
        for band in bands:
            band.remove()
        bands.clear()
        for beta in range(3):
            fill = ax_p.fill_between(
                dist[: step + 1], stacked[: step + 1, beta], stacked[: step + 1, beta + 1], color=COLOURS[beta], lw=0
            )
            bands.append(fill)
        front_p.set_xdata([dist[step]])
        front_b.set_xdata([dist[step]])
        readout.set_text("\n".join(f"{NAMES[b]} {p[step, b]:.2f}" for b in range(3)))
        shown = bl <= dist[step]
        scatter.set_offsets(np.column_stack([bl[shown], by[shown]]))
        scatter.set_color([COLOURS[f] for f in bf[shown]])

    fig.tight_layout()
    fig.subplots_adjust(right=0.86)
    return save_gif(fig, update, frame_indices(len(dist), 72), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'energy': 1.0,
    'l_max': 40000.0,
    'flavour': 'mu',
    'theta12': 33.4,
    'theta13': 8.6,
    'theta23': 49.0,
    'delta_cp': 195.0,
    'dm21': 7.4,
    'dm31': 2.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.neutrino_oscillations")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.neutrino_oscillations.main")
problem_dir = Path("problems/nuclear/neutrino_oscillations")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
